# KHUDA 10기 ML세션
Date : 2026.08.19 (수)

In [59]:
import numpy as np
import pandas as pd

In [60]:
### 건들지마세요
n=4
answer = [0] * 5

## Task 1

사과 150원짜리 4개, 귤 300원짜리 3개를 구입하고 소비세 5%가 붙는 상황을
MulLayer, AddLayer 클래스로 직접 구현해 순전파(forward)를 계산하세요.

최종 지불 금액(price)을 정수형으로, answer[1]에 저장하세요.

In [61]:
## Input Box

ans = 0

class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None

    def forward(self, x, y):
        self.x, self.y = x, y

        out = x * y

        return out

    def backward(self, dout):
        dx = dout * self.y
        dy = dout * self.x

        return dx, dy

class AddLayer:
    def forward(self, x, y):
        return x + y
    def backward(self, dout):
        return dout, dout

apple = 150
apple_num = 4
orange = 300
orange_num = 3
tax = 1.05

mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()

apple_price = mul_apple_layer.forward(apple, apple_num)
orange_price = mul_orange_layer.forward(orange, orange_num)
all_price = add_apple_orange_layer.forward(apple_price, orange_price)
price = mul_tax_layer.forward(all_price, tax)

ans = int(price)

In [62]:
answer[1] = ans
print(answer[1])

1575


## Task 2

Task 1과 동일한 계산 그래프에서 역전파를 수행하세요.
(Task 1의 계층 인스턴스를 그대로 재사용합니다.)


dprice = 1에서 시작해서 "귤 개수(orange_num)"에 대한 최종 지불 금액의 미분값(dorange_num)을 구하세요.

구체적인 구현 방식은 교재 p.164의 #역전파 부분 코드를 참고합니다.

dorange_num을 정수형으로, answer[2]에 저장하세요.


In [63]:
## Input Box

ans = 0

dprice = 1
dall_price, dtax = mul_tax_layer.backward(dprice)
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)

ans = int(dorange_num)

In [64]:
answer[2] = ans
print(answer[2])

315


## Task 3

교재 p.194~를 참고하여 예시 함수 f(x, y) = x²/20 + y² 에 대해
SGD, Momentum, AdaGrad 세 가지 옵티마이저를 모두 구현합니다(Adam은 구현되어있음).

- SGD:      lr=0.8
- Momentum: lr=0.08, momentum=0.9
- AdaGrad:  lr=1.2
- Adam:     lr=0.25, beta1=0.9, beta2=0.999

네 옵티마이저 모두 시작점 (x, y) = (-7.0, 2.0)에서 30 스텝 학습시킨 뒤,
각 옵티마이저의 최종 지점에서 원점까지의 거리 dist = sqrt(x² + y²) 를 구하세요.

dist 값 중 가장 작은 값(가장 원점에 가깝게 수렴한 옵티마이저의 dist)을
소수 첫째 자리까지 구해(소수 둘째 자리에서 반올림), 해당 값을 answer[3]에 저장하세요.

## Hint
 .values() 는 딕셔너리에서 value(dist 값)들만 꺼내는 메서드입니다. 이를 활용하면 조건에 맞는 값을 찾아낼 수 있습니다.
 예시) max(dict.values()) 는 해당 딕셔너리에서 가장 큰 값을 갖는 dist를 반환합니다.

In [65]:
## Input Box
ans = 0

def f(x, y):
    return x**2 / 20.0 + y**2

def df(x, y):
    return x / 10.0, 2.0 * y

class SGD:
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, params, grads):
        for k in params.keys():
            params[k] -= self.lr * grads[k]

class Momentum:
    def __init__(self, lr=0.01, momentum=0.9):
        self.lr = lr
        self.momentum = momentum
        self.v = None

    def update(self, params, grads):
        if self.v is None:
            self.v = {}
            for k in params.keys():
                self.v[k] = np.zeros_like(params[k])


        for k in params.keys():
            self.v[k] = self.momentum * self.v[k] - self.lr * grads[k]
            params[k] += self.v[k]

class AdaGrad:
    def __init__(self, lr=0.01):
        self.lr = lr
        self.h = None

    def update(self, params, grads):
        if self.h is None:
            self.h = {}
            for k in params.keys():
                self.h[k] = np.zeros_like(params[k])

        for k in params.keys():
            self.h[k] += grads[k] * grads[k]
            params[k] -= self.lr * grads[k] / (np.sqrt(self.h[k]) + 1e-7)

class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.iter = 0
        self.m = None
        self.v = None

    def update(self, params, grads):
        if self.m is None:
            self.m = {}
            self.v = {}
            for k in params.keys():
                self.m[k] = np.zeros_like(params[k])
                self.v[k] = np.zeros_like(params[k])
        self.iter += 1
        lr_t = self.lr * np.sqrt(1.0 - self.beta2**self.iter) / (1.0 - self.beta1**self.iter)
        for k in params.keys():
            self.m[k] += (1 - self.beta1) * (grads[k] - self.m[k])
            self.v[k] += (1 - self.beta2) * (grads[k]**2 - self.v[k])
            params[k] -= lr_t * self.m[k] / (np.sqrt(self.v[k]) + 1e-7)

def run(optimizer, steps=30):
    params = {'x': np.array(-7.0), 'y': np.array(2.0)}
    for _ in range(steps):
        gx, gy = df(params['x'], params['y'])
        optimizer.update(params, {'x': gx, 'y': gy})
    return params['x'], params['y']

## 이 밑부분 빈 값을 채우고, 옵티마이저 4개를 모두 실행해 비교해주세요
optimizers = {
    "sgd": SGD(lr=0.8),
    "momentum": Momentum(lr=0.08),
    "adagrad": AdaGrad(lr=1.2),
    "adam": Adam(lr=0.25),
}

results = {}
for name, opt in optimizers.items():
    x, y = run(opt, steps=30)
    results[name] = float(np.sqrt(x**2 + y**2))

ans = round(min(results.values()), 1)

In [66]:
answer[3] = ans
print(answer[3])

0.4


## Task 4

교재 p.246를 참고하여, Convolution 클래스를 직접 구현하세요.

Convolution.forward(x): 필터 W(FN, C, FH, FW), 편향 b, stride, pad를 받아
im2col로 입력을 펼친 뒤 행렬곱으로 합성곱 결과를 계산하고,
(N, FN, OH, OW) 형태로 복원하여 반환합니다.

출력 feature map의 모든 원소를 더한 값을 answer[4]에 저장하세요.


## Hint 전체 과정은 다음과 같습니다
1. 입력 데이터(4D) -> 2D 행렬 변환
2. 필터(4D) -> 2D 행렬 변환 및 전치(Transpose)
3. 행렬 곱 연산 + 편향 추가
4. 2D 결과를 다시 원래 4D 이미지 형태 (N, FN, OH, OW)로 복원

In [67]:
## Input Box
ans = 0


"""
참고용 : 아래 im2col 함수는 아래와 같은 방식으로 작동합니다.
    4차원 이미지 데이터를 2차원 행렬로 변환 (합성곱 연산의 속도 향상)

    [입력 데이터 Shape 변환 흐름]
    1. input_data : (N, C, H, W)
       - N: 배치 크기, C: 채널 수, H: 높이, W: 너비
    2. img (패딩 적용) : (N, C, H + 2*pad, W + 2*pad)
    3. col (임시 변환) : (N, C, filter_h, filter_w, out_h, out_w)
       - 필터가 이동하는 위치별로 영역 데이터를 수집
    4. transpose & reshape : (N * out_h * out_w, C * filter_h * filter_w)
       - 최종 출력: 2차원 행렬로 펼쳐서 행렬곱(dot product) 준비 완료
"""


def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    N, C, H, W = input_data.shape
    out_h = (H + 2*pad - filter_h)//stride + 1
    out_w = (W + 2*pad - filter_w)//stride + 1
    img = np.pad(input_data, [(0,0),(0,0),(pad,pad),(pad,pad)], mode='constant')
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w))
    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]
    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col

## 이 밑부분부터 구현해주세요
class Convolution:
    def __init__(self, W, b, stride=1, pad=0):
        self.W = W
        self.b = b
        self.stride = stride
        self.pad = pad

    def forward(self, x):
        FN, C, FH, FW = self.W.shape
        N, C, H, W = x.shape

        out_h = int((H + 2*self.pad - FH)/self.stride + 1)
        out_w = int((W + 2*self.pad - FW)/self.stride + 1)

        col = im2col(x, FH, FW, self.stride, self.pad)
        col_W = self.W.reshape(FN, -1).T
        out = np.dot(col, col_W) + self.b

        out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)
        return out

## 이 부분은 건들지 마세요.
x = np.array([[[
    [1, 2, 0, 1],
    [0, 1, 2, 1],
    [1, 0, 1, 2],
    [2, 1, 0, 1]
]]], dtype=float)   # shape (1,1,4,4)

W = np.array([[[
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1]
]]], dtype=float)   # shape (1,1,3,3)
b = np.array([0.0])

conv = Convolution(W, b, stride=1, pad=0)
out = conv.forward(x)
print(out.shape)   # (1, 1, 2, 2)

ans = round(float(np.sum(out)), 2)

(1, 1, 2, 2)


In [68]:
answer[4] = ans
print(answer[4])

20.0


# 정답 확인!

In [69]:
print("=== 작성하신 정답 ===")
for i in range (1, (n+1)) : print("Task " + str(i) + " :: " + str(answer[i]))

=== 작성하신 정답 ===
Task 1 :: 1575
Task 2 :: 315
Task 3 :: 0.4
Task 4 :: 20.0


In [70]:
### 정답 채점 코드
import hashlib

ANSWER_URL = "https://github.com/chaegyeong/KHUDA_ML_10th/raw/3c4c4e6123f5d797c13d0a68c03fb72055c5bc74/KHUDA_ML10th_WEEK6/answer.csv"

def md5 (s) : return hashlib.md5(s.encode("utf-8")).hexdigest()

ans_df = pd.read_csv(ANSWER_URL)
gt = {int(r["task"]): str(r["answer"]).strip().lower()
      for _, r in ans_df.iterrows()}

wrong = []

for i in range(1, (n+1)) :
    user_answer = "" if answer[i] is None else md5(str(answer[i]).strip().lower())
    if (user_answer != gt.get(i, "")) : wrong.append(i)

print("탈출!" if not wrong else f"틀린번호 : {wrong}")

탈출!
